In [7]:
from dataclasses import dataclass


from config.global_config import SENTIMENT_LABELS, TRAIN_ASPECTS


@dataclass
class TrainingConfig:
    data_path: str = "statics/datasets/training.csv"
    test_path: str = "statics/datasets/training.csv"
    # Podzial zbioru na zbiór treningowy i walidacyjny
    val_ratio: float = 0.2
    
    seed: int = 42
    
    learning_rate: float = 2e-5
    epochs: int = 5
    
    # Ustawienia są zależne od sprzętu
    batch_size: int = 8
    pin_memory: bool = True
    num_workers: int = 0

    @property
    def num_labels(self) -> int:
        return len(TRAIN_ASPECTS) * len(SENTIMENT_LABELS)

config = TrainingConfig()

In [8]:
import pandas as pd

df = pd.read_csv(config.data_path)
df.head(5)


,annotation_id,annotator,attractions,category,cleanliness,costs,created_at,heritage,id,infrastructure,...,lead_time,longitude,name,nature,other,rating,safety,text,time,updated_at
0,1,2,notmentioned,Park,negative,notmentioned,2026-04-03T18:08:05.212014Z,notmentioned,1,notmentioned,...,29.005,-73.881244,Sergeant Johnson Triangle,notmentioned,negative,1.0,negative,This park is dirty and full of bums sleeping a...,1501888368696,2026-04-03T18:08:05.212029Z
1,2,2,notmentioned,Park,negative,notmentioned,2026-04-03T18:08:49.682261Z,notmentioned,2,notmentioned,...,29.873,-73.881244,Sergeant Johnson Triangle,notmentioned,notmentioned,1.0,negative,Just a dirty place for dirty people to sunbathe,1528186734627,2026-04-03T18:08:49.682277Z
2,3,2,notmentioned,Parking garage,notmentioned,notmentioned,2026-04-03T18:09:15.153045Z,notmentioned,3,notmentioned,...,21.515,-73.954636,NYC Parking Manhattan Avenue. Garage Corporation.,notmentioned,positive,5.0,neutral,Attendant very helpful and courteous,1620044495060,2026-04-03T18:09:15.153061Z
3,4,2,notmentioned,Parking garage,notmentioned,positive,2026-04-03T18:10:13.194366Z,notmentioned,4,notmentioned,...,22.963,-73.954636,NYC Parking Manhattan Avenue. Garage Corporation.,notmentioned,positive,4.0,neutral,Generally friendly service and reasonable prices,1483947084522,2026-04-03T18:10:13.194395Z
4,5,2,notmentioned,Parking garage,notmentioned,positive,2026-04-03T18:10:39.551301Z,notmentioned,5,neutral,...,22.431,-73.954636,NYC Parking Manhattan Avenue. Garage Corporation.,notmentioned,notmentioned,4.0,notmentioned,They probably have the most affordable prices ...,1514987223466,2026-04-03T18:10:39.551319Z


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   annotation_id   1000 non-null   int64  
 1   annotator       1000 non-null   int64  
 2   attractions     997 non-null    str    
 3   category        1000 non-null   str    
 4   cleanliness     999 non-null    str    
 5   costs           1000 non-null   str    
 6   created_at      1000 non-null   str    
 7   heritage        998 non-null    str    
 8   id              1000 non-null   int64  
 9   infrastructure  1000 non-null   str    
 10  latitude        1000 non-null   float64
 11  lead_time       1000 non-null   float64
 12  longitude       1000 non-null   float64
 13  name            1000 non-null   str    
 14  nature          998 non-null    str    
 15  other           1000 non-null   str    
 16  rating          1000 non-null   float64
 17  safety          1000 non-null   str    
 18  

In [10]:
from config.global_config import TRAIN_ASPECTS


df[TRAIN_ASPECTS] = df[TRAIN_ASPECTS].fillna("notmentioned")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   annotation_id   1000 non-null   int64  
 1   annotator       1000 non-null   int64  
 2   attractions     1000 non-null   str    
 3   category        1000 non-null   str    
 4   cleanliness     1000 non-null   str    
 5   costs           1000 non-null   str    
 6   created_at      1000 non-null   str    
 7   heritage        1000 non-null   str    
 8   id              1000 non-null   int64  
 9   infrastructure  1000 non-null   str    
 10  latitude        1000 non-null   float64
 11  lead_time       1000 non-null   float64
 12  longitude       1000 non-null   float64
 13  name            1000 non-null   str    
 14  nature          1000 non-null   str    
 15  other           1000 non-null   str    
 16  rating          1000 non-null   float64
 17  safety          1000 non-null   str    
 18  

In [11]:
import numpy as np

def create_mapping_dicts():
    """
    Tworzy słowniki do zamiany nazw tekstowych na indeksy liczbowe.
    """
    aspect_to_idx = {asp: i for i, asp in enumerate(TRAIN_ASPECTS)}
    idx_to_aspect = {i: asp for i, asp in enumerate(TRAIN_ASPECTS)}
    
    sentiment_to_idx = {sent: i for i, sent in enumerate(SENTIMENT_LABELS)}
    idx_to_sentiment = {i: sent for i, sent in enumerate(SENTIMENT_LABELS)}
    
    return aspect_to_idx, idx_to_aspect, sentiment_to_idx, idx_to_sentiment

def generate_absa_vectors(df: pd.DataFrame) -> np.ndarray:
    """
    Konwertuje DataFrame z tagami (po Label Studio) na wektory binarne dla BERTa.
    Zwraca tablicę numpy o kształcie (liczba_opinii, 32).
    """
    aspect_to_idx, _, sentiment_to_idx, _ = create_mapping_dicts()
    
    # Długość wektora to 8 aspektów * 4 stany sentymentu = 32
    vector_length = len(TRAIN_ASPECTS) * len(SENTIMENT_LABELS)
    
    all_vectors = []
    
    # Iterujemy po każdym wierszu (opinii) w Twoim DataFrame
    for index, row in df.iterrows():
        # Inicjalizujemy wektor wypełniony zerami (długość 32)
        vec = [0] * vector_length
        
        # Sprawdzamy każdy z 8 aspektów dla tej opinii
        for aspect in TRAIN_ASPECTS:
            # Pobieramy sentyment z kolumny (np. "positive" lub "notmentioned")
            sentiment = row[aspect] 
            
            # Zabezpieczenie przed brakiem danych (jeśli zapomniałeś użyć fillna)
            if pd.isna(sentiment):
                sentiment = "notmentioned"
                
            # Obliczamy matematyczną pozycję w wektorze
            # Wzór: (indeks_aspektu * 4) + indeks_sentymentu
            idx_a = aspect_to_idx[aspect]
            idx_s = sentiment_to_idx[sentiment]
            
            position = (idx_a * len(SENTIMENT_LABELS)) + idx_s
            
            # Wstawiamy "1" na wyliczonej pozycji (One-Hot Encoding)
            vec[position] = 1
            
        all_vectors.append(vec)
        
    return np.array(all_vectors)

vectors = generate_absa_vectors(df)
    
print("Kształt macierzy wyjściowej:", vectors.shape)
print("Ilość wierszy, dlugość wektora")
print(vectors[0])

Kształt macierzy wyjściowej: (1000, 32)
Ilość wierszy, dlugość wektora
[0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 1 0]


In [12]:
df["one_hot_vector"] = list(generate_absa_vectors(df))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

from transformers import BertModel, BertTokenizer, set_seed

from config.global_config import BASE_BERT_MODEL
from model.model import BertForABSA
from model.prepare_dataset import ABSADataset
from model.train import train

# Wybrane modele do tokenizacji i prawidlowego
tokenizer = BertTokenizer.from_pretrained(BASE_BERT_MODEL)
bert = BertModel.from_pretrained(BASE_BERT_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7386.23it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
dataset = ABSADataset(df, tokenizer, label_col='one_hot_vector')

Tokenizer step:
Finished! 1000 examples, label shape torch.Size([1000, 32])



/Users/bartoszbugla/projects/projekt-magisterski/model/prepare_dataset.py:37: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  self.labels = torch.tensor(labels, dtype=torch.float32)


In [15]:
set_seed(config.seed)

val_size: int = max(1, int(len(dataset) * config.val_ratio))
train_size: int = len(dataset) - val_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_dataloader = DataLoader(
    train_ds, 
    batch_size=config.batch_size, 
    shuffle=True, 
    num_workers=config.num_workers,
    pin_memory=config.pin_memory
)

val_dataloader = DataLoader(
    val_ds, 
    batch_size=config.batch_size, 
    shuffle=False, 
    num_workers=config.num_workers,
    pin_memory=config.pin_memory
)

In [16]:
# Oczekiwany wynik to: (Batch_size, Sequence_length), (Batch_size, num_labels, czyli SENTIMENT_LABELS * TRAIN_ASPECTS)
for input, label in train_dataloader:

    print(f"Input IDs shape: {input['input_ids'].shape}")

    print(f"Labels shape: {label.shape}")

    break



/Users/bartoszbugla/projects/projekt-magisterski/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Input IDs shape: torch.Size([8, 128])
Labels shape: torch.Size([8, 32])


In [17]:
from model.train import calculate_pos_weights

# Calculate class weights from training vectors to handle imbalance
# This makes the model pay MORE attention to rare classes (positive, neutral, negative)
# and LESS attention to the dominant class (notmentioned)
pos_weights = calculate_pos_weights(vectors, device='cpu')

print("Pos weights per output (first 8 = safety sentiments):")
print(f"  positive: {pos_weights[0]:.2f}, neutral: {pos_weights[1]:.2f}, negative: {pos_weights[2]:.2f}, notmentioned: {pos_weights[3]:.2f}")

model = BertForABSA(bert)
    
history = train(
    model, 
    train_dataloader, 
    val_dataloader,
    learning_rate=config.learning_rate,
    epochs=config.epochs,
    pos_weights=pos_weights  # <-- This is the key change!
)

 20%|██        | 100/500 [00:30<01:46,  3.75it/s]


Epoch 1/5 | Train Loss: 0.4263 | Train Acc: 0.8691 | Val Loss: 0.2772 | Val Acc: 0.9322 | Time: 31.9s


 40%|████      | 200/500 [01:00<01:18,  3.82it/s]


Epoch 2/5 | Train Loss: 0.2586 | Train Acc: 0.9297 | Val Loss: 0.2319 | Val Acc: 0.9322 | Time: 29.5s


 60%|██████    | 300/500 [01:27<00:48,  4.12it/s]


Epoch 3/5 | Train Loss: 0.2282 | Train Acc: 0.9299 | Val Loss: 0.2086 | Val Acc: 0.9322 | Time: 28.0s


 80%|████████  | 400/500 [01:57<00:26,  3.82it/s]


Epoch 4/5 | Train Loss: 0.2089 | Train Acc: 0.9304 | Val Loss: 0.1941 | Val Acc: 0.9336 | Time: 29.0s


100%|██████████| 500/500 [02:26<00:00,  3.40it/s]


Epoch 5/5 | Train Loss: 0.1986 | Train Acc: 0.9316 | Val Loss: 0.1896 | Val Acc: 0.9331 | Time: 28.1s

Total training time: 146.9s


In [ ]:
import json

history_path = os.path.join(MODEL_DIR, "training_history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"Historia treningu → {history_path}")

In [18]:
# Save the trained model
import os

MODEL_DIR = f"./statics/models/absa_{len(df)}"
os.makedirs(MODEL_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'model.pt'))

In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from model.choose_architecture import choose_architecture
from model.predict import predict

device = choose_architecture()

new_model = BertForABSA(bert)
tokenizer = BertTokenizer.from_pretrained(BASE_BERT_MODEL)
new_model.load_state_dict(torch.load("statics/models/absa_1000/model.pt", map_location=device))
new_model.to(device)
new_model.eval()

texts = [
    "It is unsafe",
    "These guys damaged my car and never bothered to inform me.",
]

for text in texts:
    results, probs = predict(text, new_model, tokenizer)
    print(f"\n>>> {text}")
    print("Results:", results)
    print("Probabilities:", probs)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: mps


ValueError: too many values to unpack (expected 2, got 12)

In [ ]:
%load_ext autoreload
%autoreload 2
import torch
from model.choose_architecture import choose_architecture
from model.predict import predict

device = choose_architecture()

new_model = BertForABSA(bert)
tokenizer = BertTokenizer.from_pretrained(BASE_BERT_MODEL)
new_model.load_state_dict(torch.load("statics/models/absa_1000/model.pt", map_location=device))
new_model.to(device)
new_model.eval()

texts = [
    "It is unsafe",
    "These guys damaged my car and never bothered to inform me.",
]

for text in texts:
    results, probs = predict(text, new_model, tokenizer)
    print(f"\n>>> {text}")
    print("Results:", results)
    print("Probabilities:", probs)

Using device: mps
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: mps

>>> It is unsafe
Results: {'safety': 'notmentioned', 'cleanliness': 'notmentioned', 'infrastructure': 'notmentioned', 'nature': 'notmentioned', 'attractions': 'notmentioned', 'heritage': 'notmentioned', 'costs': 'notmentioned', 'other': 'notmentioned'}
Probabilities: {'safety': {'positive': 0.0606, 'neutral': 0.0551, 'negative': 0.0739, 'notmentioned': 0.8776}, 'cleanliness': {'positive': 0.0725, 'neutral': 0.0499, 'negative': 0.0632, 'notmentioned': 0.851}, 'infrastructure': {'positive': 0.0646, 'neutral': 0.0612, 'negative': 0.0684, 'notmentioned': 0.8304}, 'nature': {'positive': 0.0426, 'neutral': 0.0657, 'negative': 0.0562, 'notmentioned': 0.8883}, 'attractions': {'positive': 0.0568, 'neutral': 0.0644, 'negative': 0.0587, 'notmentioned': 0.8818}, 'heritage': {'positive': 0.048, 'neutral': 0.0601, 'negative': 0.0495, 'notmentioned': 0.8964}, 'costs': {'positiv